In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sharankumare/mcq-test/test.csv
/kaggle/input/datasets/sharankumare/mcq-train/train.csv


In [2]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

In [3]:
import pandas as pd

train = pd.read_csv("/kaggle/input/datasets/sharankumare/mcq-train/train.csv")

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

train.loc[150, "label"]

np.int64(2)

In [4]:
option_b_input = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

len(option_b_input)

407

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

options = ["A", "B", "C", "D", "E"]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in options
]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].reshape(1, 5, 128)

print(input_ids.shape)
print(input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])
5


In [6]:
batch = train.iloc[:16]

batch_inputs = []

for _, row in batch.iterrows():
    for option in ["A", "B", "C", "D", "E"]:
        text = str(row["prompt"]) + " [SEP] " + str(row[option])
        batch_inputs.append(text)

encoded = tokenizer(
    batch_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].reshape(16, 5, 128)

total_token_positions = input_ids.numel()

print(input_ids.shape)
print(total_token_positions)

torch.Size([16, 5, 128])
10240


In [7]:
import torch
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

row = train.loc[0]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    key: value.reshape(1, 5, 128)
    for key, value in encoded.items()
}

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits.shape)
print(outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
5


In [8]:
label = torch.tensor([train.loc[0, "label"]], dtype=torch.long)

with torch.no_grad():
    outputs = model(
        **inputs,
        labels=label
    )

loss = outputs.loss

print(loss)
print(loss.shape)
print(loss.dim())

tensor(1.5944)
torch.Size([])
0


In [9]:
!pip uninstall -y torchao
!pip install -q -U peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [10]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


295681


In [11]:
from datasets import Dataset

train_100 = train.iloc[:100].copy()

dataset = Dataset.from_pandas(train_100)

def tokenize_mcq(example):
    formatted_inputs = [
        str(example["prompt"]) + " [SEP] " + str(example[option])
        for option in ["A", "B", "C", "D", "E"]
    ]

    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": example["label"]
    }

tokenized_dataset = dataset.map(tokenize_mcq)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

first_item = tokenized_dataset[0]

print(first_item["input_ids"].shape)
print(first_item["input_ids"].shape[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

torch.Size([5, 128])
5


In [12]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer

train_32 = train.iloc[:32].copy()

dataset_32 = Dataset.from_pandas(train_32)

def tokenize_mcq_64(example):
    formatted_inputs = [
        str(example["prompt"]) + " [SEP] " + str(example[option])
        for option in ["A", "B", "C", "D", "E"]
    ]

    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": example["label"]
    }

tokenized_32 = dataset_32.map(tokenize_mcq_64)

tokenized_32.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

tiny_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

tiny_model = get_peft_model(
    tiny_model,
    lora_config
)
training_args = TrainingArguments(
    output_dir="./tiny_lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    report_to="none"
)

trainer = Trainer(
    model=tiny_model,
    args=training_args,
    train_dataset=tokenized_32
)

train_result = trainer.train()

print(trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss


4


In [13]:
import torch

row = train.loc[0]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

inputs = {
    key: value.unsqueeze(0).to(tiny_model.device)
    for key, value in encoded.items()
}

tiny_model.eval()

with torch.no_grad():
    outputs = tiny_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)

option_e_probability = probabilities[0, 4].item()

print("Logits:", outputs.logits)
print("Probabilities:", probabilities)
print("Option E probability:", option_e_probability)
print("Rounded answer:", round(option_e_probability, 4))

Logits: tensor([[-0.0247, -0.0361, -0.0745, -0.0620, -0.0782]], device='cuda:0')
Probabilities: tensor([[0.2061, 0.2038, 0.1961, 0.1986, 0.1954]], device='cuda:0')
Option E probability: 0.19538676738739014
Rounded answer: 0.1954
